# Lab: Exploring the Network Layer with `traceroute`
**Course:** Introduction to Networking  
**Topic:** IP Addressing, CIDR, Routing, and Autonomous Systems  
**Tools:** `traceroute` / `tracert`, `ipinfo.io`, Python (`requests`)

---

## Overview

In this lab, we use `traceroute` to observe how IP packets travel across the internet from our machine to various destinations. Along the way, we'll:

- Observe **IP addressing and TTL** mechanics in action
- Identify **CIDR blocks** and infer subnet boundaries
- Spot **Autonomous System (AS) boundaries** where BGP routing takes over
- Explain anomalies like `* * *` hops
- Automate IP lookups with Python

---

## Background

### How `traceroute` Works

`traceroute` works by sending packets with increasing **TTL (Time To Live)** values, starting at 1. Each router along the path decrements the TTL by 1. When TTL reaches 0, the router discards the packet and sends back an **ICMP Time Exceeded** message — revealing its IP address and the round-trip time (RTT).

```
Your Machine  →  Router 1  →  Router 2  →  ...  →  Destination
  TTL=1    ← ICMP reply ←
  TTL=2              ← ICMP reply ←
  TTL=3                         ← ICMP reply ←
```

Each line of output corresponds to one **hop** — one router in the path.

### Autonomous Systems (AS)

The internet is divided into **Autonomous Systems** — large networks operated by ISPs, universities, and companies (e.g., Comcast, Google, MIT). Routing *between* ASes uses **BGP (Border Gateway Protocol)**. In a traceroute, you can often see the moment traffic crosses from one AS to another by watching IP address blocks change.

### CIDR Notation

IP addresses are grouped into blocks using CIDR notation: `192.168.1.0/24` means the first 24 bits are the network prefix, leaving 8 bits for hosts (256 addresses). Recognizing CIDR blocks helps you tell whether consecutive hops are within the same network or not.

---

## Part 1: Running `traceroute`

### Step 1.1 — Choose Your Targets

We'll trace routes to four destinations that represent different types of hosts:

| # | Target | Why |
|---|--------|-----|
| A | `google.com` | Large CDN, well-peered |
| B | A university (e.g. `mit.edu`) | Academic network, different AS |
| C | An international host (e.g. `bbc.co.uk`) | Cross-continental routing |
| D | Your choice | Pick anything interesting |

Feel free to swap targets — the analysis process is the same regardless.

### Step 1.2 — Run the Commands

**On Mac/Linux (Terminal):**
```bash
traceroute google.com
traceroute mit.edu
traceroute bbc.co.uk
```

**On Windows (CMD):**
```cmd
tracert google.com
tracert mit.edu
tracert bbc.co.uk
```

> 💡 **Tip:** Add `-m 30` (Mac/Linux) or `-h 30` (Windows) to allow up to 30 hops if the default cuts off early.

### Step 1.3 — Record Your Output

Paste your raw `traceroute` output into the cell below for each target. Keep the formatting as-is.

#### Target A: `google.com`

```
tracert google.com

Tracing route to google.com [2607:f8b0:4006:801::200e]
over a maximum of 30 hops:

  1     4 ms     4 ms     3 ms  2601:19b:701:8110:deeb:69ff:fe4d:38
  2    16 ms    13 ms    12 ms  2001:558:10f2:218c::2
  3    17 ms    16 ms    14 ms  po-62-rur201.westroxbury.ma.boston.comcast.net [2001:558:200:20e1::1]
  4    15 ms    17 ms    17 ms  po-2-rur202.westroxbury.ma.boston.comcast.net [2001:558:200:2dc::2]
  5    18 ms    15 ms    18 ms  po-300-xar02.westroxbury.ma.boston.comcast.net [2001:558:200:813b::1]
  6    17 ms    17 ms    12 ms  be-334-ar01.needham.ma.boston.comcast.net [2001:558:200:2044::1]
  7    16 ms     *        *     be-502-arsc1.needham.ma.boston.comcast.net [2001:558:200:39d::1]
  8    23 ms    18 ms    19 ms  be-32031-cs03.newyork.ny.ibone.comcast.net [2001:558:3:282::1]
  9     *        *       17 ms  be-3311-pe11.111eighthave.ny.ibone.comcast.net [2001:558:3:86::2]
 10    21 ms    21 ms    22 ms  2607:f8b0:8461:180::1
 11    20 ms    21 ms    22 ms  2607:f8b0:8461:180::1
 12    21 ms    21 ms    22 ms  2001:4860:0:1::1c7e
 13    19 ms    16 ms    19 ms  2001:4860:0:1::5f77
 14    22 ms    21 ms    21 ms  pnlgaa-az-in-x0e.1e100.net [2607:f8b0:4006:801::200e]
```

#### Target B: `mit.edu`

```
tracert mit.edu

Tracing route to mit.edu [2600:141b:1c00:389::255e]
over a maximum of 30 hops:

  1     6 ms     5 ms     5 ms  2601:19b:701:8110:deeb:69ff:fe4d:38
  2    16 ms    16 ms    16 ms  2001:558:10f2:218c::3
  3    16 ms    13 ms    13 ms  po-62-rur202.westroxbury.ma.boston.comcast.net [2001:558:200:20e2::1]
  4    25 ms    17 ms    23 ms  po-300-xar02.westroxbury.ma.boston.comcast.net [2001:558:200:813b::1]
  5     *       17 ms     *     be-334-ar01.needham.ma.boston.comcast.net [2001:558:200:2044::1]
  6     *        *       16 ms  be-502-arsc1.needham.ma.boston.comcast.net [2001:558:200:39d::1]
  7    25 ms    18 ms    21 ms  be-32031-cs03.newyork.ny.ibone.comcast.net [2001:558:3:282::1]
  8    23 ms    21 ms    23 ms  be-2211-pe11.newark.nj.ibone.comcast.net [2001:558:3:7d::2]
  9    24 ms    22 ms    20 ms  2001:559:0:19::6
 10    26 ms    25 ms    23 ms  ae35.r02.border101.ewr04.fab.netarch.akamai.com [2600:1488:a240::19]
 11    22 ms    26 ms    50 ms  vlan102.r05.spine101.ewr04.fab.netarch.akamai.com [2600:141b:6000:306::1]
 12    23 ms    22 ms    23 ms  vlan105.r04.leaf101.ewr04.fab.netarch.akamai.com [2600:141b:6000:a04::1]
 13    22 ms    22 ms    20 ms  vlan104.r08.tor101.ewr04.fab.netarch.akamai.com [2600:141b:6000:2908::1]
 14    17 ms    19 ms    21 ms  g2600-141b-1c00-0389-0000-0000-0000-255e.deploy.static.akamaitechnologies.com [2600:141b:1c00:389::255e]
```

#### Target C: `bbc.co.uk`

```
tracert bbc.co.uk

Tracing route to bbc.co.uk [2a04:4e42:400::81]
over a maximum of 30 hops:

  1     4 ms     4 ms     4 ms  2601:19b:701:8110:deeb:69ff:fe4d:38
  2    16 ms    14 ms    17 ms  2001:558:10f2:218c::2
  3    21 ms    60 ms    24 ms  po-62-rur201.westroxbury.ma.boston.comcast.net [2001:558:200:20e1::1]
  4    17 ms    13 ms    13 ms  po-2-rur202.westroxbury.ma.boston.comcast.net [2001:558:200:2dc::2]
  5    21 ms    13 ms    13 ms  po-300-xar02.westroxbury.ma.boston.comcast.net [2001:558:200:813b::1]
  6    16 ms     *       16 ms  be-334-ar01.needham.ma.boston.comcast.net [2001:558:200:2044::1]
  7     *        *       18 ms  be-502-arsc1.needham.ma.boston.comcast.net [2001:558:200:39d::1]
  8    14 ms    14 ms    16 ms  2001:559:e000::96
  9    16 ms    17 ms    17 ms  2a04:4e42:400::81
```

#### Target D: _(your choice)_ (Choice: facebook.com)

```
  1     4 ms     4 ms     3 ms  2601:19b:701:8110:deeb:69ff:fe4d:38
  2    17 ms    13 ms    13 ms  2001:558:10f2:218c::3
  3    15 ms    12 ms    14 ms  po-62-rur202.westroxbury.ma.boston.comcast.net [2001:558:200:20e2::1]
  4    14 ms    12 ms    13 ms  po-300-xar02.westroxbury.ma.boston.comcast.net [2001:558:200:813b::1]
  5     *        *        *     Request timed out.
  6    19 ms     *       17 ms  be-502-arsc1.needham.ma.boston.comcast.net [2001:558:200:39d::1]
  7    13 ms    14 ms    17 ms  2001:558:fe03:406::16
  8    23 ms    20 ms    17 ms  po4001.asw03.bos5.tfbnw.net [2620:0:1cff:dead:beef::92cc]
  9    14 ms    17 ms    17 ms  po2103.psw01.bos5.tfbnw.net [2620:0:1cff:dead:beef::463]
 10    11 ms    14 ms    10 ms  po1.msw1ae.01.bos5.tfbnw.net [2a03:2880:f07e:ffff::9]
 11    16 ms    16 ms    17 ms  edge-star-mini6-shv-01-bos5.facebook.com [2a03:2880:f380:1:face:b00c:0:25de]
```

---

## Part 2: Manual IP Analysis

Before automating anything, let's build intuition by looking up a few hops manually.

### Step 2.1 — Pick 6–8 Hops to Investigate

From your combined traceroute output, select 6–8 interesting IP addresses — try to pick a mix of:
- Early hops (likely your ISP)
- Middle hops (backbone/transit networks)
- Late hops (destination network)
- At least one hop right before or after a `* * *`

### Step 2.2 — Look Them Up

For each IP, go to **[https://ipinfo.io/\<IP\>](https://ipinfo.io)** (e.g. `https://ipinfo.io/8.8.8.8`) and record:
- **Organization / ASN** (e.g. `AS15169 Google LLC`)
- **CIDR block** (listed as `prefix`, e.g. `8.8.8.0/24`)
- **Country / Region**

Fill in the table below:

### Manual Lookup Table

| Hop # | IP Address | Organization | ASN | CIDR Block | Location |
|-------|------------|--------------|-----|------------|---------|
|1| 2601:19b:701:8110:deeb:69ff:fe4d:38|Comcast | 	AS7922  |2601::/20 |Boston, MA |
|5|2001:558:200:813b::1| Comcast| 	AS7922 |2001:558::/29 | Boston, MA |
|7 |2001:558:200:39d::1 |Comcast |AS7922 | 	2001:558::/29 | Cambridge, MA|
|10 |2607:f8b0:8461:180::1 |Google |AS15169 | 2607:f8b0::/32| New York City, New York|
| 13| 2001:4860:0:1::5f77 |Google | 	AS15169 | 	2001:4860::/32 |New York City, New York |
|14 |2607:f8b0:4006:801::200e |Google |AS15169 |2607:f8b0:4006::/48 |Mountain View, CA |
|4 |2001:558:200:813b::1 |Comcast |AS7922 |2001:558::/29 |Boston, MA |
|6 |2001:558:200:39d::1 |Comcast |AS7922 |  	2001:558::/29| Cambridge, MA |

traceroute used is google.com's traceroute. 
exception is last two hops, which is facebook.com's traceroute.

> ✏️ **Quick analysis (answer in 2–3 sentences):** Looking at your table, can you spot where traffic moves from your ISP's network into a larger backbone or transit network? How can you tell from the ASN or CIDR block?

The IP address generally has the most significant bits jump to a different base when the network changes. with the ASN block, it changes as well, and with the CIDR block, it helps with the bit prefix.

---

## Part 3: Automated IP Lookup with Python

Doing this manually for every hop gets tedious fast. Let's automate it using the free `ipinfo.io` API.

### Step 3.1 — Install dependencies (if needed)

The `requests` library is used to make HTTP calls to the API. It's included in most Python environments, but run this if needed:

In [2]:
# Run this cell only if 'requests' is not already installed
%pip install requests

Note: you may need to restart the kernel to use updated packages.


### Step 3.2 — Define Your Hops

Paste in all the IP addresses from **one** of your traceroutes below. Skip any `* * *` hops (they have no IP to look up).

In [4]:
# Replace these with the actual IPs from one of your traceroutes
# Skip hops that showed * * * (no IP available)

hops = [
    # (hop_number, ip_address)
    (1,  "2601:19b:701:8110:deeb:69ff:fe4d:38"),
    (2,  "2001:558:10f2:218c::2"),
    (3,  "2001:558:200:20e1::1"),
    (4,  "2001:558:200:2dc::2"),
    (5,  "2001:558:200:813b::1"),
    (6,  "2001:558:200:2044::1"),
    (7,  "2001:558:200:39d::1"),
    (8,  "2001:558:3:282::1"),
    (9,  "2001:558:3:86::2"),
    (10,  "2607:f8b0:8461:180::1"),
    (11,  "2607:f8b0:8461:180::1"),
    (12,  "2001:4860:0:1::1c7e"),
    (13,  " 2001:4860:0:1::5f77"),
    (14,  "2607:f8b0:4006:801::200e"),
    # add more as needed...
]

# Which traceroute target did these hops come from?
target = "google.com" 

### Step 3.3 — Query the API and Display Results

In [5]:
import requests
import time

def lookup_ip(ip):
    """Query ipinfo.io for details about an IP address."""
    try:
        response = requests.get(f"https://ipinfo.io/{ip}/json", timeout=5)
        data = response.json()
        return {
            "ip":      data.get("ip", ip),
            "org":     data.get("org", "N/A"),       # includes ASN, e.g. "AS15169 Google LLC"
            "city":    data.get("city", "N/A"),
            "country": data.get("country", "N/A"),
        }
    except Exception as e:
        return {"ip": ip, "org": "lookup failed", "city": "", "country": ""}


print(f"Traceroute analysis for: {target}")
print("-" * 75)
print(f"{'Hop':<5} {'IP Address':<18} {'ASN / Organization':<35} {'Location'}")
print("-" * 75)

results = []
for hop_num, ip in hops:
    info = lookup_ip(ip)
    results.append((hop_num, info))
    location = f"{info['city']}, {info['country']}" if info['city'] else info['country']
    print(f"{hop_num:<5} {info['ip']:<18} {info['org']:<35} {location}")
    time.sleep(0.3)  # be polite to the free API

print("-" * 75)

Traceroute analysis for: google.com
---------------------------------------------------------------------------
Hop   IP Address         ASN / Organization                  Location
---------------------------------------------------------------------------
1     2601:19b:701:8110:deeb:69ff:fe4d:38 AS7922 Comcast Cable Communications, LLC Boston, US
2     2001:558:10f2:218c::2 AS7922 Comcast Cable Communications, LLC Mount Laurel, US
3     2001:558:200:20e1::1 AS7922 Comcast Cable Communications, LLC Boston, US
4     2001:558:200:2dc::2 AS7922 Comcast Cable Communications, LLC Boston, US
5     2001:558:200:813b::1 AS7922 Comcast Cable Communications, LLC Boston, US
6     2001:558:200:2044::1 AS7922 Comcast Cable Communications, LLC Boston, US
7     2001:558:200:39d::1 AS7922 Comcast Cable Communications, LLC Cambridge, US
8     2001:558:3:282::1  AS7922 Comcast Cable Communications, LLC New York City, US
9     2001:558:3:86::2   AS7922 Comcast Cable Communications, LLC New York City, U

### Step 3.4 — Detect AS Boundary Crossings

Now let's automatically flag where the Autonomous System changes between consecutive hops — these are the BGP handoff points.

In [6]:
def extract_asn(org_string):
    """Pull the ASN out of a string like 'AS15169 Google LLC'."""
    if org_string and org_string.startswith("AS"):
        return org_string.split(" ")[0]  # e.g. 'AS15169'
    return None


print("AS Boundary Crossings Detected:")
print("=" * 50)

prev_asn = None
prev_hop = None
boundaries_found = 0

for hop_num, info in results:
    current_asn = extract_asn(info["org"])
    
    if current_asn and prev_asn and current_asn != prev_asn:
        print(f"  Hop {prev_hop} → Hop {hop_num}: {prev_asn} → {current_asn}")
        print(f"    (BGP handoff: traffic crosses into a new Autonomous System)")
        boundaries_found += 1
    
    if current_asn:
        prev_asn = current_asn
        prev_hop = hop_num

if boundaries_found == 0:
    print("  No AS boundaries detected (try a longer or more international route).")
else:
    print(f"\n  Total AS boundaries crossed: {boundaries_found}")

AS Boundary Crossings Detected:
  Hop 9 → Hop 10: AS7922 → AS15169
    (BGP handoff: traffic crosses into a new Autonomous System)

  Total AS boundaries crossed: 1


---

## Part 4: Analysis and Discussion

Answer the following questions based on your results. Aim for 2–4 sentences each.

---

### Q1 — TTL and Hop Count

How many hops did your longest traceroute take? What does this tell you about the path complexity between you and that destination? Is the number of hops what you expected?

THe longest traceroute was to mit and google, both at 14 hops. I think this tells me the more AS crossings happen, the more complex it takes to go between destinations. I did not expect the number of hops to be larger than the bbc.

### Q2 — The `* * *` Mystery

Did any hops show `* * *` instead of an IP and RTT? Explain, using your understanding of ICMP and TTL, why this happens. Is it always a problem, or can traffic still flow through these hops?

I believe this is because the trace was not properly sent for too long causing a TTL drop. I am guessing the traffic will attempt to send it through a different channel while attempting to reestablish the conenction.

### Q3 — CIDR and Subnet Boundaries

Look at consecutive hops that belong to the same organization (same ASN). Do their IP addresses fall within the same CIDR block? Give a specific example from your data and compute how many host addresses that CIDR block could contain.

Some of them fall within the same CIDR block, but not all of them. Comcast specifically had the 2001:558::/29 block, and there would be 128 - 29 = 99, 2^99 addresses.

### Q4 — Autonomous Systems and BGP

How many distinct Autonomous Systems did your traffic pass through (across all four traces)? Describe one AS boundary crossing: which two organizations were involved, and what does it mean in practice that routing handed off between them?

I believe there is really about one AS jump per each of the four traces. One of the crossings was between Comcast and Google, and in practice it means Comcast believed the AS that would best get my trace through and also is allowed to send to was Google, and also made sure it wasn't self looping. Then it send my data there to continue the trace. 

### Q5 — Route Asymmetry (Bonus)

If you run `traceroute` to the same destination twice in a row, do you always get the same path? Try it and observe. Why might the route differ, and what does this reveal about how routing decisions are made on the internet?

Depending on traffic and existing connections, an optimal route (depending on how optimal is defined through BGPs, etc) can be different. My second trace to facebook ended up skipping an internal connection as it had two request time outs, and it chose a different more direct route instead of through a intermediary connection.

---

## Appendix: Concept Reference

| Term | Definition |
|------|------------|
| **TTL** | Time To Live — decremented by each router; packet discarded (and ICMP reply sent) when it hits 0 |
| **ICMP** | Internet Control Message Protocol — used by `traceroute` for Time Exceeded and Echo Reply messages |
| **CIDR** | Classless Inter-Domain Routing — e.g. `192.168.0.0/16` denotes a block of 65,536 addresses |
| **ASN** | Autonomous System Number — unique ID for each AS (e.g. AS15169 = Google) |
| **BGP** | Border Gateway Protocol — the routing protocol used between Autonomous Systems |
| **RTT** | Round-Trip Time — the time for a packet to reach a hop and for the reply to come back |
| **`* * *`** | Hop that did not return an ICMP reply (firewall blocked it, or router configured not to respond) |